1. Pydantic定义的对象，langchain处理成提示词的方式,JsonOutputParser, PydanticOutputParser得到系统提示词，注入到模型调用
2. 直接给模型JSON schema


# 基础准备

1. model的结构化输出[models#structured-output](https://docs.langchain.com/oss/python/langchain/models#structured-output)

In [1]:
import os

import dotenv

dotenv.load_dotenv()
os.environ["base_url"]

'https://api.siliconflow.cn/v1'

通过具体的[langchain models api](https://docs.langchain.com/oss/python/langchain/models)

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

# Pydantic

In [6]:
from pydantic import BaseModel, Field


class Prime(BaseModel):
    prime: list[int] = Field(description="素数")
    count: list[int] = Field(description="小于该素数的素数个数")

# JsonOutputParser得到系统提示词

JsonOutputParser根据Pydantic的定义结构得到系统提示词

In [8]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser(pydantic_object=Prime)

In [ ]:
# 得到系统提示词
system_prompt = json_parser.get_format_instructions()

In [12]:
print(system_prompt)

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [14]:
result_1 = model.invoke(
    [
        ("system", system_prompt),
        ("user", "请提供前10个素数及其对应的小于该素数的素数个数。"),
    ]
)

In [19]:
type(result_1)

langchain_core.messages.ai.AIMessage

In [16]:
result_1.pretty_print()

================================== Ai Message ==================================

{"prime":[2,3,5,7,11,13,17,19,23,29],"count":[0,1,2,3,4,5,6,7,8,9]}


In [20]:
result_1.content

'{"prime":[2,3,5,7,11,13,17,19,23,29],"count":[0,1,2,3,4,5,6,7,8,9]}'

## JsonOutputParser解析得到dict

In [22]:
result_json_1 = json_parser.invoke(result_1.content)

In [ ]:
print(type(result_json_1), result_json_1)

<class 'dict'> {'prime': [2, 3, 5, 7, 11, 13, 17, 19, 23, 29], 'count': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}


## PydanticOutputParser解析得到原始类

In [28]:
from langchain_core.output_parsers import PydanticOutputParser

pydantic_parser = PydanticOutputParser(pydantic_object=Prime)

In [31]:
result_json_11 = pydantic_parser.invoke(result_1.content)
print(type(result_json_11), result_json_11)

<class '__main__.Prime'> prime=[2, 3, 5, 7, 11, 13, 17, 19, 23, 29] count=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


直接访问属性

In [32]:
result_json_11.prime

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

# 模型产生支持JSON Schema


当你调用 `model.with_structured_output(Prime)` 时，LangChain 底层会根据你使用的模型和 `method` 参数，选择一种“传递结构”的方式：

**方式一：`json_schema`（通常是默认）**
LangChain 把 `Prime` 转成 JSON Schema，通过 API 的 `response_format` 参数传给模型。这是 OpenAI 等厂商的原生结构化输出功能，模型被**强制**按照这个 Schema 生成内容，不符合就会失败。这种方式下，Schema **不是**作为“工具”传入的，而是作为“响应格式”约束。

**方式二：`function_calling`（工具调用）**
LangChain 把 `Prime` 转成一个“函数/工具”的定义，这个定义里包含了参数结构（也就是你 Pydantic 模型的字段和类型）。模型看到有一个工具可以调用，并且 LangChain 通常会设置 `tool_choice` 来强制模型调用这个工具。模型“调用”工具时，实际上就是在输出符合你 Schema 的 JSON 参数。

所以，“作为 Tools 传入”描述的是**方式二**。你之前看到的“获取不到系统提示词”的现象，在方式一下同样成立——因为结构约束根本不在对话消息里，而在 API 的元参数中。

In [33]:
model_with_structure = model.with_structured_output(Prime)

In [45]:
result_2 = model_with_structure.invoke(
    "请提供前10个素数及其对应的小于该素数的素数个数。"
)

In [46]:
result_2

Prime(prime=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], count=[2, 3, 5, 7, 11, 13, 17, 19, 23, 29])

In [47]:
print(type(result_2), result_2.prime)

<class '__main__.Prime'> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [48]:
import json

print(json.dumps(Prime.model_json_schema(), indent=2, ensure_ascii=False))

{
  "properties": {
    "prime": {
      "description": "素数",
      "items": {
        "type": "integer"
      },
      "title": "Prime",
      "type": "array"
    },
    "count": {
      "description": "小于该素数的素数个数",
      "items": {
        "type": "integer"
      },
      "title": "Count",
      "type": "array"
    }
  },
  "required": [
    "prime",
    "count"
  ],
  "title": "Prime",
  "type": "object"
}
